# Module 5b - NetOps log analysis with `ai_query` (bulk inference)

**Time**: 20 minutes

## What you'll do

Take real, public network-operator outage reports (the `outages@outages.org` mailing list archive) and extract structured fields from every message in **one SQL statement** using `ai_query`. Then expose the result to the compound agent so it can answer questions like *"any recent BGP outages affecting Tier-1 carriers?"* with real evidence.

## Why this notebook exists

The Knowledge Assistant in Module 4 retrieves doctrine (STIGs, RFCs, advisories). It does not aggregate. `ai_query` is the right tool when you have **structured extraction at scale** — the same prompt applied to every row of a table. This is the pattern a NOC analyst wants for triage email digests, ticketing system dumps, syslog forwarders, change request feeds, anything text-shaped at volume.

## Data source

`https://puck.nether.net/pipermail/outages/` — monthly `.txt.gz` mbox files containing every message sent to the `outages` mailing list. Real network operators describing real incidents in plain English.

In [ ]:
%run ./_config


## 1. Pull recent monthly archives + parse the mbox

In [ ]:
import requests, gzip, io, mailbox, re, email
from email.utils import parsedate_to_datetime
from datetime import datetime, timezone

BASE_URL = "https://puck.nether.net/pipermail/outages"

# Discover the actually available monthly archives by parsing the index page —
# the archive isn't strictly contiguous (volunteer-run list).
idx = requests.get(f"{BASE_URL}/", timeout=30).text
available = re.findall(r'href="([0-9]{4}-[A-Z][a-z]+\.txt\.gz)"', idx)
print(f"index lists {len(available)} monthly archives; most recent: {available[:3]}")

TARGET = 12  # how many of the most recent monthly archives to pull
archives = []
for fname in available[:TARGET]:
    url = f"{BASE_URL}/{fname}"
    r = requests.get(url, timeout=30)
    if r.status_code == 200 and len(r.content) > 200:
        archives.append((fname.replace('.txt.gz', ''), r.content))
        print(f"  + {fname} ({len(r.content)} bytes)")
print(f"\nPulled {len(archives)} monthly archives")


In [ ]:
def strip_quoted(text: str) -> str:
    """Drop quoted-reply lines and signatures so ai_query sees only original prose."""
    out = []
    for line in text.splitlines():
        if line.startswith(">"):
            continue
        if line.strip() == "--":
            break
        out.append(line)
    return "\n".join(out).strip()

rows = []
for label, gz_bytes in archives:
    text = gzip.decompress(gz_bytes).decode("utf-8", errors="ignore")
    # Parse from string by splitting on the standard "From " envelope line
    msgs = []
    cur = []
    for line in text.splitlines(keepends=True):
        if line.startswith("From ") and cur:
            msgs.append("".join(cur))
            cur = []
        cur.append(line)
    if cur:
        msgs.append("".join(cur))
    for raw in msgs:
        try:
            m = email.message_from_string(raw)
        except Exception:
            continue
        subject = (m.get("Subject") or "").strip()
        sender = (m.get("From") or "").strip()
        date_raw = m.get("Date")
        try:
            sent_at = parsedate_to_datetime(date_raw).astimezone(timezone.utc) if date_raw else None
        except Exception:
            sent_at = None
        # Body: walk parts, prefer text/plain
        body = ""
        if m.is_multipart():
            for p in m.walk():
                if p.get_content_type() == "text/plain":
                    body = p.get_payload(decode=True).decode("utf-8", errors="ignore")
                    break
        else:
            payload = m.get_payload(decode=True) or b""
            body = payload.decode("utf-8", errors="ignore")
        clean = strip_quoted(body)[:4000]
        if not subject or not clean:
            continue
        rows.append({
            "archive_month": label,
            "message_id": (m.get("Message-ID") or "").strip("<>"),
            "in_reply_to": (m.get("In-Reply-To") or "").strip("<>"),
            "sender": sender,
            "sent_at": sent_at,
            "subject": subject[:300],
            "body": clean,
        })

print(f"Parsed {len(rows)} messages.")
rows[0] if rows else None

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema = StructType([
    StructField("archive_month", StringType(), True),
    StructField("message_id", StringType(), True),
    StructField("in_reply_to", StringType(), True),
    StructField("sender", StringType(), True),
    StructField("sent_at", TimestampType(), True),
    StructField("subject", StringType(), True),
    StructField("body", StringType(), True),
])
df = spark.createDataFrame(rows, schema=schema)
df = df.dropDuplicates(["message_id"]).filter(F.col("message_id") != "")
df.write.mode("overwrite").saveAsTable(f"{BASE}.netops_outage_messages")
print(f"Wrote {df.count()} rows to {BASE}.netops_outage_messages")

## 2. The bulk-inference move: one `ai_query` extracts structured fields from every message

This is the punchline of the module. Notice: one SQL statement, one prompt, every row processed in parallel by Mosaic AI Model Serving. No Python loop, no batch driver code.

In [ ]:
import json
extract_schema = json.dumps({
    "type": "json_schema",
    "json_schema": {
        "name": "outage_extract",
        "schema": {
            "type": "object",
            "properties": {
                "is_outage_report":   {"type": "boolean", "description": "True if the message describes an actual network outage or impairment (vs a follow-up, tooling question, off-topic, or social)."},
                "root_cause_category":{"type": "string", "enum": ["bgp", "fiber_cut", "dns", "power", "routing", "ddos", "peering", "hardware", "config_change", "software_bug", "capacity", "third_party", "unknown", "other"]},
                "affected_providers": {"type": "array", "items": {"type": "string"}, "description": "Carrier, ISP, cloud provider, or AS names mentioned as affected."},
                "affected_regions":   {"type": "array", "items": {"type": "string"}, "description": "Geographic regions or specific cities/markets mentioned."},
                "severity":           {"type": "string", "enum": ["low", "medium", "high", "critical"]},
                "status":             {"type": "string", "enum": ["new_report", "ongoing", "resolved", "follow_up", "unknown"]},
                "summary":            {"type": "string", "description": "One sentence describing the incident in operator-facing language."},
            },
            "required": ["is_outage_report", "root_cause_category", "severity", "summary"],
        },
        "strict": True,
    },
}).replace("'", "''")

PROMPT = (
    "You are a NOC analyst. Read this message from the public outages mailing list "
    "and extract structured incident attributes. Be conservative: if the message is a follow-up, "
    "a question, or off-topic, set is_outage_report=false. Use the message subject and body together. "
    "Return JSON only.\n\nSubject: "
)

spark.sql(f"""
    CREATE OR REPLACE TABLE {BASE}.netops_outages_structured AS
    SELECT
      message_id, archive_month, sender, sent_at, subject,
      ai_query(
        '{LLM_HAIKU}',
        CONCAT('{PROMPT}', subject, '\\n\\nBody:\\n', LEFT(body, 3500)),
        responseFormat => '{extract_schema}'
      ) AS extract
    FROM {BASE}.netops_outage_messages
""")

spark.sql(f"SELECT COUNT(*) AS rows FROM {BASE}.netops_outages_structured").display()

## 3. Flatten the JSON into queryable columns

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE VIEW {BASE}.netops_outages AS
    SELECT
      message_id,
      sent_at,
      subject,
      sender,
      parse_json(extract):is_outage_report::boolean        AS is_outage_report,
      parse_json(extract):root_cause_category::string      AS root_cause,
      parse_json(extract):severity::string                 AS severity,
      parse_json(extract):status::string                   AS status,
      parse_json(extract):summary::string                  AS summary,
      from_json(parse_json(extract):affected_providers::string, 'ARRAY<STRING>') AS affected_providers,
      from_json(parse_json(extract):affected_regions::string,   'ARRAY<STRING>') AS affected_regions
    FROM {BASE}.netops_outages_structured
""")

spark.sql(f"SELECT * FROM {BASE}.netops_outages WHERE is_outage_report = true ORDER BY sent_at DESC LIMIT 10").display()

## 4. Aggregate the way a NOC dashboard would

In [ ]:
spark.sql(f"""
    SELECT root_cause, severity, COUNT(*) AS n
    FROM {BASE}.netops_outages
    WHERE is_outage_report = true
    GROUP BY root_cause, severity
    ORDER BY n DESC
""").display()

In [ ]:
# Top providers mentioned in outage reports
spark.sql(f"""
    SELECT provider, COUNT(*) AS mentions
    FROM {BASE}.netops_outages
    LATERAL VIEW EXPLODE(affected_providers) AS provider
    WHERE is_outage_report = true
    GROUP BY provider
    ORDER BY mentions DESC
    LIMIT 15
""").display()

## 5. Register a SQL function the compound agent can call

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE FUNCTION {BASE}.recent_outages(provider_query STRING, days INT)
    RETURNS TABLE
    COMMENT 'Return outage reports from the last N days mentioning the given provider/carrier (case-insensitive substring match). Pass provider_query=\"\" to return all recent outages.'
    RETURN
      SELECT sent_at, subject, root_cause, severity, status, summary, affected_providers, affected_regions
      FROM {BASE}.netops_outages
      WHERE is_outage_report = true
        AND sent_at >= date_sub(current_timestamp(), recent_outages.days)
        AND (recent_outages.provider_query = ''
             OR exists(affected_providers,
                       p -> lower(p) LIKE lower(concat('%', recent_outages.provider_query, '%'))))
      ORDER BY sent_at DESC
""")
print(f"Created {BASE}.recent_outages(provider_query, days)")


In [ ]:
# Persist for module 5 to discover via _workshop_config (so the agent picks it up automatically)
cfg_set('netops_outages_view', f'{BASE}.netops_outages')
cfg_set('netops_recent_outages_fn', f'{BASE}.recent_outages')

## What just happened

1. We pulled real, public outage reports — no synthetic data.
2. We ran a **single SQL statement** with `ai_query` over every row, in parallel, on serverless compute. The same prompt produced structured JSON for every message.
3. We exposed the result as a view **and** a UC function. The compound agent (Module 5) automatically discovers the function through `_workshop_config` and gains a fifth tool: *"any recent BGP outages affecting Lumen?"*

## The pattern

| Use case | Source | `ai_query` extracts |
|---|---|---|
| Outage triage | mailing list, ticket dump | this notebook's schema |
| Change-request risk grading | ServiceNow CR table | risk_class, blast_radius, rollback_plan_present |
| Syslog noise reduction | Splunk / OpenSearch | event_class, severity, host_role |
| Vendor PSIRT triage | RSS/Atom feeds | affected_models, exploitability, has_workaround |

Every one of those is the same structural pattern: one prompt + a JSON schema, applied to a column.